# Seguridad del streaming

SAIF aplicado al pipeline real de streaming: `scripts/stream_producer.py` firma cada
evento con HMAC antes de mandarlo a Redpanda, y `tp_mlops2.streaming.consumer` verifica
esa firma y el esquema de rangos antes de predecir. Los controles viven en
`src/tp_mlops2/streaming/security.py`, no en este notebook: aca se prueban directamente
las funciones reales que usan el productor y el consumidor en produccion.

In [1]:
import random

from tp_mlops2.streaming.security import (
    DLQ, EventoDemandaValido, chequear_alerta_dlq, consumir_validado, firmar,
)

random.seed(0)


def evento_real(i: int, envenenado: bool = False) -> dict:
    """Misma forma que arma scripts/stream_producer.py."""
    ev = {
        "timestamp": f"2018-06-{(i % 28) + 1:02d}T{(i % 24):02d}:00:00",
        "hour": i % 24, "dow": i % 7, "month": 6, "is_weekend": int((i % 7) >= 5),
        "temp_Madrid": round(random.gauss(25, 5), 1),
        "temp_Barcelona": round(random.gauss(23, 4), 1),
        "temp_Valencia": round(random.gauss(24, 4), 1),
        "temp_Seville": round(random.gauss(30, 5), 1),
        "temp_Bilbao": round(random.gauss(18, 4), 1),
        "load_lag_24h": round(random.gauss(28000, 3000), 1),
        "load_lag_168h": round(random.gauss(28000, 3000), 1),
    }
    if envenenado:
        ev["temp_Madrid"] = 900.0           # temperatura fisicamente imposible
        ev["load_lag_24h"] = -50000.0       # demanda negativa, imposible
    return ev


stream = [evento_real(i) for i in range(8)] + [evento_real(99, envenenado=True)]
print(f"Stream de prueba: {len(stream)} eventos, el ultimo envenenado")

Stream de prueba: 9 eventos, el ultimo envenenado


## 1. Firma HMAC: el productor real firma, el consumidor real verifica

Cada evento que manda `stream_producer.py` incluye su firma (`firmar()` de
`security.py`). Si la firma no coincide, el consumidor lo rechaza directo, antes de
mirar siquiera el contenido.

In [2]:
aceptados = []
for i, ev in enumerate(stream):
    # el evento 8 lo mandamos sin firmar correctamente, simulando un productor no autorizado
    firma_valida = firmar(ev) if i != 8 else "firma-falsa-de-un-productor-no-autorizado"
    resultado = consumir_validado(ev, firma_valida)
    if resultado is not None:
        aceptados.append(resultado)

print(f"Aceptados: {len(aceptados)} de {len(stream)}")
print(f"En cuarentena (DLQ): {len(DLQ)}")
for d in DLQ:
    print(" -", d["motivo"])

Aceptados: 8 de 9
En cuarentena (DLQ): 1
 - firma HMAC invalida


## 2. Firma correcta, contenido envenenado: el esquema lo frena igual

El `EventoDemandaValido` de `security.py` valida los rangos fisicamente plausibles del
dataset real (ver `notebooks/01_dataset_EDA.ipynb`). Una firma valida no alcanza si el
contenido esta fuera de rango.

In [3]:
evento_envenenado = evento_real(99, envenenado=True)
firma_correcta = firmar(evento_envenenado)  # el productor "autorizado" firma algo invalido

resultado = consumir_validado(evento_envenenado, firma_correcta)
print("resultado:", resultado, "(None: la validacion de esquema lo freno igual)")
print("\nDLQ actualizada:")
for d in DLQ:
    print(" -", d["motivo"])

resultado: None (None: la validacion de esquema lo freno igual)

DLQ actualizada:
 - firma HMAC invalida
 - esquema invalido: Input should be less than or equal to 50


## 3. Alerta por volumen de cuarentena

`chequear_alerta_dlq` es la misma funcion que usa `tp_mlops2.streaming.consumer` en cada
ventana de 20 eventos: si la cantidad en DLQ llega al umbral, dispara una alerta de
posible productor comprometido.

In [4]:
from tp_mlops2.streaming.security import UMBRAL_ALERTA_DLQ

print(f"umbral configurado: {UMBRAL_ALERTA_DLQ} eventos en cuarentena por ventana")
if chequear_alerta_dlq(DLQ):
    print(f"ALERTA: {len(DLQ)} eventos en cuarentena en esta ventana "
          "- posible ataque de envenenamiento o productor comprometido en curso")
else:
    print(f"{len(DLQ)} eventos en cuarentena - dentro de lo esperado")

umbral configurado: 2 eventos en cuarentena por ventana
ALERTA: 2 eventos en cuarentena en esta ventana - posible ataque de envenenamiento o productor comprometido en curso


## Reflexion

Estas no son funciones de ejemplo: son las que importa `stream_producer.py` para firmar
y las que importa `tp_mlops2.streaming.consumer` para validar y disparar la alerta de
DLQ en cada ventana, corriendo ahora mismo dentro del pipeline real. Un evento que no
viene de un productor autorizado, o que trae un valor fisicamente imposible, nunca llega
a generar una prediccion.